# RiskMatch · From a client goal to an allocation review

Run from the repository root. This public walkthrough uses the web app’s interview engine and fictional market observations. The mandate and market example use USD. Standardized lottery cards use CHF. Use the [checklist](CHECKLIST.md) for completion criteria and the [research notebook](RiskMatch_research_replays.ipynb) for local historical experiments.

## 1. Profile one goal

Exercise a reproducible swipe route. Simulated responses test the software only. The human pilot will measure comprehension, time and the stability of the risk-aversion interval.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from riskmatch.intake import sample
from riskmatch.profile_session import Interview

facts = dict(sample(), currency="USD", wish="4-6")
session = Interview(facts, seed=42)
for _ in range(200):
    stage, key = session.current()
    if stage == "review":
        break
    direction = "right"
    if key.startswith("A"):
        # Simulate a consistent preference at phi = 3 across randomized statements.
        preferred = int(3 < session.cards[key]["phi_threshold"])
        direction = "right" if session.statement()["right"] == preferred else "left"
    session.apply(dict(action="swipe", id=key, step=session.step,
                       direction=direction, seconds=1))
assert session.current()[0] == "review"
report = session.report()
capacity = report["capacity"]
print("Cards answered:", report["completion"], "Tolerance:", report["tolerance"])
print({k: capacity[k] for k in ("currency", "reserve", "free", "desired_return", "required_rate")})

## 2. Register the evidence and freeze a decision date

Inspect the pre-existing hypotheses. The independent tests, macro vintages, forecast intervals and admission decisions are tracked in [the research protocol](RESEARCH.md). Only the following synthetic three-asset example runs here.

In [ ]:
registry = pd.read_csv("experiments.csv")
display(registry[["id", "target", "benchmark", "primary_metric", "sample_status"]])
bundle = json.loads(Path("data/demo/market_bundle.json").read_text())
assert bundle["synthetic"] and capacity["currency"] == "USD"
frame = bundle["frames"]["prices"]
prices = pd.DataFrame(frame["values"], columns=frame["columns"],
                      index=pd.to_datetime(frame["dates"]))[["SPY", "IEF", "GLD"]]
assert prices.index.is_unique and prices.index.is_monotonic_increasing
assert prices.notna().all().all() and (prices > 0).all().all()
returns = prices.pct_change(fill_method=None).dropna()
train, review = returns.iloc[:-20], returns.iloc[-20:]
assert len(train) >= 20 and train.index.max() < review.index.min()
print("Last training close:", train.index.max(), "Review close:", review.index.max())

## 3. Compare attainable risk and net return

Illustrate a long-only grid in 10% increments for the investable sleeve. Use annual arithmetic means and sample covariance. Charge 10 bp on the initial purchase and assume no further trades over the one-year estimation horizon. This coarse frontier illustrates the calculation. Real admission requires mandate screening, dated forecasts, trading frictions and independent tests.

In [ ]:
from itertools import product

fee = 0.001
annual_mean = train.mean() * 252
cov = train.cov() * 252
rows = []
for equity, bond in product(range(11), repeat=2):
    if equity + bond > 10:
        continue
    w = np.array([equity, bond, 10 - equity - bond]) / 10
    net = (1 + w @ annual_mean) / (1 + fee) - 1
    risk = np.sqrt(w @ cov @ w) / (1 + fee)
    rows.append([*w, net, risk])
grid = pd.DataFrame(rows, columns=[*prices.columns, "net_mean", "vol"])
ordered = grid.sort_values(["vol", "net_mean"], ascending=[True, False])
frontier = ordered[ordered.net_mean > ordered.net_mean.cummax().shift(fill_value=-np.inf)]
eligible = frontier[frontier.vol <= capacity["risk_cap_pct"] / 100]
assert np.allclose(grid[prices.columns].sum(axis=1), 1)
display(eligible)
score = report["tolerance"]
if eligible.empty or score["lo"] is None or score["hi"] is None:
    print("Personal ranking withheld: no eligible row or no bounded tolerance interval.")
else:
    for phi in (score["lo"], score["hi"]):
        utility = eligible.net_mean - phi * eligible.vol.pow(2) / 2
        print("Aversion:", phi, "highest-utility grid row:", utility.idxmax())

## 4. Review a holding at a later close

Use an explicit equal-weight control to check ledger arithmetic independently of the personal ranking. Keep the reserve in cash with zero interest. Mark drift and estimate the cost of selling and buying back to target. A real proposal needs confirmed consent, dated execution quotes and a reconciled post-trade ledger.

In [ ]:
weights = np.full(3, 1 / 3)
initial = capacity["free"]
reserve = capacity["reserve"]
units = initial * weights / (1 + fee) / prices.loc[train.index[-1]].to_numpy()
values = prices.loc[review.index].mul(units, axis=1)
wealth = values.sum(axis=1) + reserve
last = values.iloc[-1]
target = last.sum() * weights
estimated_fee = (target - last).abs().sum() * fee
assert np.isclose((units * prices.loc[train.index[-1]]).sum() * (1 + fee), initial)
display(pd.DataFrame({"target": weights, "observed": last / last.sum()}))
print("Portfolio including reserve:", round(wealth.iloc[-1], 2),
      "estimated trading fee:", round(estimated_fee, 2))
wealth.rename("Synthetic portfolio value including reserve").plot(ylabel="USD")

## Evidence still needed

Run participant calibration, release-date checks, out-of-sample forecast evaluation, constrained net backtests and catalogue review before exposing ranked options to clients. The web app currently ends at profile confirmation. Monitoring here is a dated simulation.